<a href="https://colab.research.google.com/github/esrakirbas/msc-thesis-reducing-hallucinations-in-medical-diagnosis/blob/main/code/python/encoding_patient_and_encounter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
%pip install -q \
    chromadb \
    "opentelemetry-api==1.42.1" \
    "opentelemetry-sdk==1.42.1" \
    "opentelemetry-exporter-otlp-proto-grpc==1.42.1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.0 MB/s eta 0:00:00


In [4]:
import pandas as pd
import chromadb
from chromadb.errors import NotFoundError

# file is on 'main' branch (public repo)
github_repo_url = "https://raw.githubusercontent.com/esrakirbas/medical-synthetic-prototype-data/main/data/"

# Read the CSV file into a pandas DataFrame
patients_url = (github_repo_url + "patients.csv")

conditions_url = (github_repo_url + "conditions.csv")

encounters_url = (github_repo_url + "encounters.csv")

observations_url = (github_repo_url + "observations.csv")

procedures_url = (github_repo_url + "procedures.csv")

allergies_url = (github_repo_url + "allergies.csv")

medications_url = (github_repo_url + "medications.csv")

## Get Chroma DB client and collection
client = chromadb.Client()
collection_name = "patient_context"
try:
    client.delete_collection(name=collection_name)
    print(f"Deleted existing collection: {collection_name}")
except NotFoundError:
    print(f"Collection did not exist: {collection_name}")
collection = client.create_collection(name=collection_name)

## PATIENTS DATA
# Read the CSV file into a pandas DataFrame
patients_df = pd.read_csv(patients_url)

#Patients CSV data rec pattern : ['Id', 'BIRTHDATE', 'DEATHDATE', 'FIRST', 'LAST', 'MARITAL', 'RACE', 'ETHNICITY', 'GENDER']
gender_map = {
    "M": "male",
    "F": "female"
}
maritial_map = {
    "M": "married",
    "S": "single",
    "D": "divorced"
}
vector_ids = []
documents = []
metadatas = []

for index, row in patients_df.iterrows():
  vector_ids.append(f"patient:{row['Id']}:profile")
  documents.append(f"Patient demographic profile: {gender_map.get(row["GENDER"], "unknown").capitalize()}, born on {row['BIRTHDATE']}, {maritial_map.get(row["MARITAL"], "Unknown")}, {(row['RACE']).capitalize()}, and {(row['ETHNICITY']).capitalize()}.")
  metadatas.append({
        "patient_id": str(row['Id']),
        "record_type": "patient_profile",
        "source_table": "patients",
        "birthdate": str(row["BIRTHDATE"]),
        "gender": gender_map.get(row["GENDER"], "unknown").capitalize(),
        "marital_status": maritial_map.get(row["MARITAL"], "Unknown"),
        "race": (row['RACE']).capitalize(),
        "ethnicity": (row['ETHNICITY']).capitalize(),
        "deceased": not pd.isna(row["DEATHDATE"])
    })

batch_size = min(250, client.get_max_batch_size())

for start in range(0, len(vector_ids), batch_size):
    end = start + batch_size

    collection.add(
        ids=vector_ids[start:end],
        documents=documents[start:end],
        metadatas=metadatas[start:end]
    )

    print(f"Added {min(batch_size,len(vector_ids))} of {len(vector_ids)} records")

print(f"Patient count : {collection.count()}")

#results = collection.query (query_texts=["single"], n_results=collection.count())
#results = collection.get(where={"marital_status": "single"})
#print(f"Query result count (single patients) : {len(results["ids"])}")
#print(results)

encounters_df = pd.read_csv(encounters_url)
vector_ids = []
documents = []
metadatas = []

for index, row in encounters_df.head(5).iterrows():
  encounter_chunk = (f"{row['ENCOUNTERCLASS']} ")
  encounter_description = (row['DESCRIPTION']).strip().lower()
  if not encounter_description.startswith("encounter"):
    encounter_chunk += "encounter for "
  encounter_chunk += encounter_description
  if pd.notna(row["REASONDESCRIPTION"]):
    encounter_chunk += ". The reason for the encounter was " + row["REASONDESCRIPTION"]

  print(encounter_chunk)


"""

# Display the first few rows of the DataFrame
print("First 3 rows of patients data...>")
display(patients_df.head(3))

#CONDITIONS DATA
print("\nFirst 3 rows of conditions data...>")
conditions_df = pd.read_csv(conditions_url)
display(conditions_df.head(3))

#ENCOUNTERS DATA
print("\nFirst 3 rows of encounters data...>")
display(encounters_df.head(3))

#OBSERVATIONS DATA
print("\nFirst 3 rows of observations data...>")
observations_df = pd.read_csv(observations_url)
display(observations_df.head(3))
print("\nTotal # of rows in observations data = " + str(len(observations_df)))

#PROCEDURES DATA
print("\nFirst 3 rows of procedures data...>")
procedures_df = pd.read_csv(procedures_url)
display(procedures_df.head(3))
print("\nTotal # of rows in procedures data = " + str(len(procedures_df)))

#ALLERGIES DATA
print("\nFirst 3 rows of allergies data...>")
allergies_df = pd.read_csv(allergies_url)
display(allergies_df.head(3))

#MEDICATIONS DATA
print("\nFirst 3 rows of medications data...>")
medications_df = pd.read_csv(medications_url)
display(medications_df.head(3))
"""

Deleted existing collection: patient_context


/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:00<00:00, 83.7MiB/s]


Added 108 of 108 records
Patient count : 108
ambulatory encounter for problem (procedure). The reason for the encounter was Glycine max (substance)
wellness encounter for general examination of patient (procedure)
ambulatory encounter for problem (procedure). The reason for the encounter was Allergic disposition (finding)
wellness encounter for well child visit (procedure)
wellness encounter for general examination of patient (procedure)


'\n\n# Display the first few rows of the DataFrame\nprint("First 3 rows of patients data...>")\ndisplay(patients_df.head(3))\n\n#CONDITIONS DATA\nprint("\nFirst 3 rows of conditions data...>")\nconditions_df = pd.read_csv(conditions_url)\ndisplay(conditions_df.head(3))\n\n#ENCOUNTERS DATA\nprint("\nFirst 3 rows of encounters data...>")\ndisplay(encounters_df.head(3))\n\n#OBSERVATIONS DATA\nprint("\nFirst 3 rows of observations data...>")\nobservations_df = pd.read_csv(observations_url)\ndisplay(observations_df.head(3))\nprint("\nTotal # of rows in observations data = " + str(len(observations_df)))\n\n#PROCEDURES DATA\nprint("\nFirst 3 rows of procedures data...>")\nprocedures_df = pd.read_csv(procedures_url)\ndisplay(procedures_df.head(3))\nprint("\nTotal # of rows in procedures data = " + str(len(procedures_df)))\n\n#ALLERGIES DATA\nprint("\nFirst 3 rows of allergies data...>")\nallergies_df = pd.read_csv(allergies_url)\ndisplay(allergies_df.head(3))\n\n#MEDICATIONS DATA\nprint("\nFir